In [1]:
import re
import html
import pandas as pd
from pathlib import Path

In [2]:
INPUT_PATH = Path("amazon_final_data.csv")
OUTPUT_DIR = Path("processed_data")
OUTPUT_PATH = OUTPUT_DIR / "processed_amazon_support.csv"

In [3]:
def fix_encoding_issues(text):
    """
    Tries to fix common weird characters like â€™ and Â£.
    If ftfy is not installed, the code still works without it.
    """
    try:
        from ftfy import fix_text
        return fix_text(text)
    except ImportError:
        return text

In [12]:
def basic_clean_text(text):
    """
    Cleaning for RAG:
    - keep the sentence readable
    - remove Twitter noise
    - do not stem
    - do not remove stopwords
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Fix HTML symbols like &amp;
    text = html.unescape(text)

    # Fix broken characters if ftfy exists
    text = fix_encoding_issues(text)

    # Replace URLs with a placeholder instead of deleting them completely
    text = re.sub(r"http\S+|www\S+", "[SUPPORT_LINK]", text)

    # Remove Twitter mentions like @AmazonHelp, @115820
    text = re.sub(r"@\w+", " ", text)

    # Remove agent signatures like ^AG, ^TN, ^KL
    text = re.sub(r"\^[A-Za-z]{1,4}\b", " ", text)

    # Remove hashtags symbol but keep the word
    text = re.sub(r"#", "", text)

    # Hide possible order numbers
    # Hide possible Amazon-like order numbers
    text = re.sub(r"\b\d{3}-\d{7}-\d{7}\b", "[ORDER_NUMBER]", text)

    # Hide long order numbers only, not normal numbers like "I order 6 items"
    text = re.sub(
        r"\border\s*#?\s*(\d[\d-]{8,}\d)\b",
        "order [ORDER_NUMBER]",
        text,
        flags=re.IGNORECASE
    )

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
def clean_for_eda(text):
    """
    More aggressive cleaning for analysis only.
    This version is not used as the final chatbot context.
    """
    text = basic_clean_text(text)

    text = text.lower()

    # Keep letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
def count_words(text):
    if pd.isna(text) or str(text).strip() == "":
        return 0
    return len(str(text).split())


In [7]:
def load_data(path):
    df = pd.read_csv(path)

    expected_columns = {"question", "answer", "company"}
    missing_columns = expected_columns - set(df.columns)

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    return df


In [8]:
def preprocess_data(df):
    # Keep only needed columns
    df = df[["question", "answer", "company"]].copy()

    # Remove exact duplicate rows
    df = df.drop_duplicates().reset_index(drop=True)

    # Remove rows with empty question or answer
    df = df.dropna(subset=["question", "answer"]).reset_index(drop=True)

    # Cleaned readable text for RAG
    df["clean_question"] = df["question"].apply(basic_clean_text)
    df["clean_answer"] = df["answer"].apply(basic_clean_text)

    # Extra-clean text for EDA only
    df["eda_question"] = df["question"].apply(clean_for_eda)
    df["eda_answer"] = df["answer"].apply(clean_for_eda)

    # Length features
    df["question_word_count"] = df["clean_question"].apply(count_words)
    df["answer_word_count"] = df["clean_answer"].apply(count_words)

    # Remove rows that became too short after cleaning
    df = df[
        (df["question_word_count"] >= 3) &
        (df["answer_word_count"] >= 3)
    ].reset_index(drop=True)

    return df

In [9]:
def print_data_report(original_df, processed_df):
    print("=" * 60)
    print("Preprocessing Report")
    print("=" * 60)

    print(f"Original rows: {len(original_df)}")
    print(f"Processed rows: {len(processed_df)}")
    print(f"Removed rows: {len(original_df) - len(processed_df)}")

    print("\nMissing values after processing:")
    print(processed_df.isnull().sum())

    print("\nQuestion word count stats:")
    print(processed_df["question_word_count"].describe())

    print("\nAnswer word count stats:")
    print(processed_df["answer_word_count"].describe())

    print("\nSample cleaned rows:")
    sample_cols = ["question", "clean_question", "answer", "clean_answer"]
    print(processed_df[sample_cols].head(5).to_string())


In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    original_df = load_data(INPUT_PATH)
    processed_df = preprocess_data(original_df)

    processed_df.to_csv(OUTPUT_PATH, index=False)

    print_data_report(original_df, processed_df)

    print("\nSaved processed file to:")
    print(OUTPUT_PATH)


if __name__ == "__main__":
    main()

Preprocessing Report
Original rows: 123561
Processed rows: 123023
Removed rows: 538

Missing values after processing:
question               0
answer                 0
company                0
clean_question         0
clean_answer           0
eda_question           0
eda_answer             0
question_word_count    0
answer_word_count      0
dtype: int64

Question word count stats:
count    123023.000000
mean         20.607699
std          10.122036
min           3.000000
25%          14.000000
50%          20.000000
75%          25.000000
max          64.000000
Name: question_word_count, dtype: float64

Answer word count stats:
count    123023.000000
mean         19.713639
std           7.629456
min           3.000000
25%          15.000000
50%          19.000000
75%          22.000000
max          55.000000
Name: answer_word_count, dtype: float64

Sample cleaned rows:
                                                                                                                      

: 